Nama: Fadhlan Nur Rachman
NIM: 2802491690

In [15]:
import pandas as pd
import os
import pickle
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy
from IPython.display import clear_output

MODEL_PATH = 'model.pickle'
CSV_PATH = 'emotion_comment.csv'

# Cek + Declaration Dataset
df = pd.read_csv(CSV_PATH)
print(df.head)

print(df['Emotion'].value_counts())

<bound method NDFrame.head of                                                 Comment Emotion
0     i seriously hate one subject to death but now ...    fear
1                    im so full of life i feel appalled   anger
2     i sit here to write i start to dig out my feel...    fear
3     ive been really angry with r and i feel like a...     joy
4     i feel suspicious if there is no one outside l...    fear
...                                                 ...     ...
5932                 i begun to feel distressed for you    fear
5933  i left feeling annoyed and angry thinking that...   anger
5934  i were to ever get married i d have everything...     joy
5935  i feel reluctant in applying there because i w...    fear
5936  i just wanted to apologize to you because i fe...   anger

[5937 rows x 2 columns]>
Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64


# Data Preprocessing

In [ ]:
def get_label(tag):
    if tag.startswith('j'):
        return 'a'
    elif tag.startswith('r') or tag.startswith('v') or tag.startswith('n'):
        return tag[0]
    else:
        return 'n'

def lemmatizing(word_list):
    lemmatizer = WordNetLemmatizer()
    lemmatized = []
    tagged = pos_tag(word_list)
    for word, tag in tagged:
        label = get_label(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemmatized.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemmatized.append(result)

    return lemmatized

def preprocessing(sentence):

    #tokenizer
    word_list = word_tokenize(sentence)

    # lowering
    word_list = [word.lower() for word in word_list]

    # Remove Stopwords and Number
    eng_stopwords = stopwords.words('english')
    removed_punc = [word for word in word_list if word not in eng_stopwords and word.isalpha()]

    # Stemming
    # stemmer = SnowballStemmer('english')
    # stemmed = [stemmer.stem(word) for word in removed_punc]

    # Lemmatizing + Post Tag
    lemmatized = lemmatizing(removed_punc)

    return lemmatized



# Debugging
sentence = df['Comment'][0]
preprocessing(sentence)

['seriously', 'hate', 'one', 'subject', 'death', 'feel', 'reluctant', 'drop']

# Feature Extraction

In [34]:
# Check FreqDist
x = df['Comment']
y = df['Emotion']

all_sentence = " ".join(x)
all_token = preprocessing(all_sentence)
freq_dist = FreqDist(all_token)

print("Top words: ", freq_dist.most_common(10))

Top words:  [('feel', 5421), ('like', 1006), ('im', 943), ('feeling', 804), ('get', 507), ('make', 429), ('go', 418), ('know', 374), ('time', 359), ('want', 351)]


In [36]:
def extract_features(comment):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in comment)
    return features


features_sets = []
for(comment, emotion) in zip(x, y):
    features = extract_features(preprocessing(comment))
    features_sets.append((features, emotion))

print(features_sets[0])

({'seriously': True, 'hate': True, 'one': True, 'subject': True, 'death': True, 'feel': True, 'reluctant': True, 'drop': True, 'im': False, 'full': False, 'life': False, 'appal': False, 'sit': False, 'write': False, 'start': False, 'dig': False, 'feeling': False, 'think': False, 'afraid': False, 'accept': False, 'possibility': False, 'might': False, 'make': False, 'ive': False, 'really': False, 'angry': False, 'r': False, 'like': False, 'idiot': False, 'trust': False, 'first': False, 'place': False, 'suspicious': False, 'outside': False, 'rapture': False, 'happen': False, 'something': False, 'jealous': False, 'becasue': False, 'want': False, 'kind': False, 'love': False, 'true': False, 'connection': False, 'two': False, 'soul': False, 'friend': False, 'mine': False, 'keep': False, 'tell': False, 'morbid': False, 'thing': False, 'dog': False, 'finally': False, 'fell': False, 'asleep': False, 'useless': False, 'still': False, 'anxiety': False, 'bit': False, 'annoyed': False, 'antsy': Fal

# Data Splitting

In [43]:
train_count = int(len(features_sets) * 0.8)
train_set = features_sets[:train_count]
test_set = features_sets[train_count:]

print("All Data: ", len(features_sets))
print("Train Data: ", len(train_set))
print("Test Data: ", len(test_set))

All Data:  5937
Train Data:  4749
Test Data:  1188


# Modelling with Naive Bayes

In [53]:
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    test_acc = accuracy(classifier, test_set)

    classifier.most_informative_features(10)
    print("Accuracy: ", test_acc*100)

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier, f)
    return classifier

In [64]:
def write_comment():
    while True:
        comment = input("Input your comment: ")

        if len(comment.split()) < 2:
            print("Your comment are word not sentence")
        else:
            return comment

def load_model():
    with open(MODEL_PATH, 'rb') as f:
        classifier = pickle.load(f)
    return classifier

def show_syn_ant(word_list):
    for word in word_list:
        synonyms = []
        antonyms = []
        synset = wordnet.synsets(word)

        for syn in synset:
            for lemma in syn.lemmas():
                synonyms.append(lemma.name())
                for ant in lemma.antonyms():
                    antonyms.append(ant.name())
        print("Word:", word)
        if synonyms:
            print("- Synonyms: ", synonyms[:5])
        else:
            print("- No Synonyms")

        if antonyms:
            print("- Antonyms: ", antonyms[:5])
        else:
            print("No Antoynyms")
        

def analyze_comment(comment, classifier):
    if comment == '':
        print("Your Comment is Null and press enter to main menu")
        input("Press Enter to Main Menu")
        return comment

    print('Original Comment: ', comment)
    word_list = preprocessing(comment)
    tagged = pos_tag(word_list)
    print("POS Tagging: ")
    for word,tag in tagged:
        print("-", word, ":", tag)

    print("\n Print Synonym & Antonym")
    show_syn_ant(word_list)

    preprocessed_comment = preprocessing(comment)
    feature_extracted = extract_features(preprocessed_comment)
    prediction = classifier.classify(feature_extracted)

    print("prediction comment: ", prediction)

    

# Debugging
# analyze_comment("hei are you so fucking, because i hate you and love you in same time")
    

# Main Menu

In [66]:
comment = ""
classifier = None


while True:
    clear_output(True)
    print("Your Comment: ", comment)
    print("1. Write your comment")
    print("2. Analyze your comment")
    print("3. Exit")
    choice = input("Input choice(1/2/3): ")

    if choice == "1":
        print("\nWrite your comment!")
        comment = write_comment()

        if os.path.exists(MODEL_PATH):
            classifier = load_model()
            print("Model Loaded from File!")
        else:
            print("Model has training again so wait until done!")
            classifier = train_model()
            print("Model Loaded from training!")

        input("Press Enter to Main Menu")
    elif choice == "2":
        print("\nAnalyze your comment!")
        analyze_comment(comment, classifier)
        input("Press Enter to Main Menu")
    elif choice == "3":
        clear_output()
        print("Exited prgram.....")
        break
    else:
        print("Please input your choice (1/2/3) and Press Enter to your Choice")
        input("Press Enter to Main Menu")
    

Exited prgram.....
